# geocode_optimized.ipynb

Optimized Lawrence, MA geocoding pipeline.

Input: `checkpoint2.csv`  
Output: `checkpoint3_geocoded.csv`  
Cache: `geocode_cache.csv`

Run the code cell below after setting `DATA_DIR` or `PROJECT_DATA_DIR` if your CSVs are not in the current notebook folder.

In [1]:
"""
geocode_optimized.py

Optimized geocoding pipeline for Lawrence, MA police incident data.

Input:  checkpoint2.csv
Output: checkpoint3_geocoded.csv
Cache:  geocode_cache.csv

Notebook usage:
    Run this file cell-by-cell in a notebook, or execute as a script after
    setting PROJECT_DATA_DIR to the folder containing checkpoint2.csv.
"""

from __future__ import annotations

import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd
import requests


In [2]:

# -----------------------------------------------------------------------------
# 1. Configuration
# -----------------------------------------------------------------------------

# By default, use the current notebook/script folder.
# Change DATA_DIR if your CSVs live somewhere else.
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = Path(os.environ.get("PROJECT_DATA_DIR", NOTEBOOK_DIR))

INPUT_CSV = Path("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/checkpoint2.csv")
OUTPUT_CSV = DATA_DIR / "checkpoint3_geocoded.csv"
CACHE_CSV = DATA_DIR / "geocode_cache.csv"
MA_BOUNDARY_CACHE = DATA_DIR / "ma_boundary.geojson"

# OpenCage key lookup order:
# 1. Environment variable OPENCAGE_API_KEY
# 2. ../config.json or ./config.json containing {"OPENCAGE_API_KEY": "..."}
CONFIG_CANDIDATES = [NOTEBOOK_DIR / "config.json", NOTEBOOK_DIR.parent / "config.json"]

OPENCAGE_URL = "https://api.opencagedata.com/geocode/v1/json"
SLEEP_SECONDS = 1.2
BATCH_SAVE_EVERY = 25
MAX_RETRIES = 5

CACHE_COLUMNS = [
    "raw_address",
    "cleaned_address",
    "lat",
    "long",
    "address_confidence",
    "geocoded_at",
]

# Official Census cartographic boundary file. The downloaded boundary is cached
# locally as ma_boundary.geojson after the first successful run.
CENSUS_STATE_BOUNDARY_URL = (
    "https://www2.census.gov/geo/tiger/GENZ2023/shp/"
    "cb_2023_us_state_500k.zip"
)



In [3]:

# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------

def load_opencage_key() -> str:
    """Load OpenCage API key from env var or config.json."""
    key = os.environ.get("OPENCAGE_API_KEY")
    if key:
        return key

    for config_path in CONFIG_CANDIDATES:
        if config_path.exists():
            with open(config_path, "r", encoding="utf-8") as f:
                config = json.load(f)
            key = config.get("OPENCAGE_API_KEY")
            if key:
                return key

    raise RuntimeError(
        "OpenCage API key not found. Set OPENCAGE_API_KEY or add it to config.json."
    )


def clean_address(address: Any) -> str:
    """
    Clean one Location value for geocoding.

    Rules:
    - Do not use Location Prefix.
    - Strip unit/floor/apartment/suite/room/trailing # info.
    - Preserve both streets in intersections.
    - Normalize common suffix variants.
    - Convert address ranges to the first street number.
    - Append ', Lawrence, MA' exactly once.
    """
    if pd.isna(address):
        return ""

    cleaned = str(address).upper().strip()
    cleaned = re.sub(r"\s+", " ", cleaned)

    # Normalize intersection separators while preserving both sides.
    cleaned = re.sub(r"\s*/\s*", " & ", cleaned)
    cleaned = re.sub(r"\s+AND\s+", " & ", cleaned)
    cleaned = re.sub(r"\s*&\s*", " & ", cleaned)

    # Remove common unit/floor/apartment/suite markers and everything after them.
    # Examples: '550 BROADWAY FL 1' -> '550 BROADWAY'; '12 OAK ST #2' -> '12 OAK ST'
    cleaned = re.sub(
        r"\s+(?:FL|FLOOR|APT|APARTMENT|UNIT|STE|SUITE|RM|ROOM|BLDG|BUILDING)\b.*$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s+#\S+.*$", "", cleaned)

    # Remove commas after the useful address component, if present.
    cleaned = cleaned.split(",")[0].strip()

    # Normalize common suffixes without destroying intersections.
    suffix_fixes = {
        "AV": "AVE",
        "AV.": "AVE",
        "AVENUE": "AVE",
        "ST.": "ST",
        "STREET": "ST",
        "RD.": "RD",
        "ROAD": "RD",
        "CT.": "CT",
        "COURT": "CT",
        "BLVD.": "BLVD",
        "DR.": "DR",
        "DRIVE": "DR",
    }
    for old, new in suffix_fixes.items():
        cleaned = re.sub(rf"\b{re.escape(old)}\b", new, cleaned)

    # Handle OCR / typing artifacts like STFL.
    cleaned = re.sub(r"\bST\s*FL\b", "ST", cleaned)
    cleaned = re.sub(r"\bSTFL\b", "ST", cleaned)

    # Convert address ranges to the first number.
    cleaned = re.sub(r"\b(\d{1,5})-\d{1,5}-\d{1,5}\b", r"\1", cleaned)
    cleaned = re.sub(r"\b(\d{1,5})-\d{1,5}\b", r"\1", cleaned)

    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if not cleaned:
        return ""
    # Always append the city/state after cleaning.
    #  This avoids mistaking LAWRENCE ST for Lawrence, MA.
    if not re.search(r",\s*LAWRENCE\s*,?\s*MA\b", cleaned, flags=re.IGNORECASE):
        cleaned = re.sub(r",?\s*(LAWRENCE,\s*)?MA$", "", cleaned, flags=re.IGNORECASE).strip()
        cleaned = f"{cleaned}, Lawrence, MA"
        return cleaned


def load_cache(cache_path: Path) -> pd.DataFrame:
    """Load or initialize geocode cache with a stable schema."""
    if cache_path.exists():
        cache = pd.read_csv(cache_path)
        for col in CACHE_COLUMNS:
            if col not in cache.columns:
                cache[col] = pd.NA
        cache = cache[CACHE_COLUMNS]
    else:
        cache = pd.DataFrame(columns=CACHE_COLUMNS)

    # Idempotency: one cache row per raw Location value.
    cache["raw_address"] = cache["raw_address"].astype(str).str.strip()
    cache = cache.drop_duplicates(subset=["raw_address"], keep="last")
    cache = cache[CACHE_COLUMNS]
    cache.to_csv(cache_path, index=False)
    return cache


def geocode_with_opencage(
    cleaned_address: str,
    api_key: str,
    sleep_seconds: float = SLEEP_SECONDS,
    max_retries: int = MAX_RETRIES,
) -> Dict[str, Optional[Any]]:
    """Geocode one cleaned address using OpenCage, with retry/backoff."""
    if not cleaned_address:
        return {"lat": None, "long": None, "address_confidence": None}

    params = {
        "q": cleaned_address,
        "key": api_key,
        "limit": 1,
        "countrycode": "us",
        "no_annotations": 1,
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(OPENCAGE_URL, params=params, timeout=30)
            if response.status_code == 402:
                raise RuntimeError("OPENCAGE_QUOTA_EXHAUSTED")
            if response.status_code in [401, 403]:
                raise RuntimeError(f"OPENCAGE_AUTH_ERROR_{response.status_code}: check your API key")
            if response.status_code == 429 or 500 <= response.status_code < 600:
                wait = min(60, (2 ** attempt) * sleep_seconds)
                print(
                    f"Transient OpenCage response {response.status_code} for "
                    f"'{cleaned_address}'. Retrying in {wait:.1f}s."
                )
                time.sleep(wait)
                continue
            if response.status_code == 402:
                raise RuntimeError("OPENCAGE_QUOTA_EXHAUSTED")
            response.raise_for_status()
            payload = response.json()
            results = payload.get("results", [])

            if not results:
                return {"lat": None, "long": None, "address_confidence": None}

            top = results[0]
            geometry = top.get("geometry", {})
            return {
                "lat": geometry.get("lat"),
                "long": geometry.get("lng"),
                "address_confidence": top.get("confidence"),
            }

        except requests.RequestException as exc:
            wait = min(60, (2 ** attempt) * sleep_seconds)
            print(f"Request error for '{cleaned_address}': {exc}. Retrying in {wait:.1f}s.")
            time.sleep(wait)

    return {"lat": None, "long": None, "address_confidence": None}


def save_cache(cache: pd.DataFrame, cache_path: Path) -> None:
    """Write deduplicated cache to disk."""
    cache = cache[CACHE_COLUMNS].drop_duplicates(subset=["raw_address"], keep="last")
    cache.to_csv(cache_path, index=False)


def load_massachusetts_boundary(data_dir: Path):
    """
    Load Massachusetts boundary as a GeoDataFrame.

    Requires geopandas. Downloads an official Census state boundary file only once,
    then caches the MA polygon locally as ma_boundary.geojson.
    """
    try:
        import geopandas as gpd
    except ImportError as exc:
        raise ImportError(
            "geopandas is required for the MA spatial boundary filter. Install with: "
            "pip install geopandas shapely pyogrio"
        ) from exc

    if MA_BOUNDARY_CACHE.exists():
        return gpd.read_file(MA_BOUNDARY_CACHE).to_crs("EPSG:4326")

    states = gpd.read_file(CENSUS_STATE_BOUNDARY_URL)
    ma = states[states["STUSPS"] == "MA"].to_crs("EPSG:4326")
    ma.to_file(MA_BOUNDARY_CACHE, driver="GeoJSON")
    return ma


def filter_to_massachusetts(df: pd.DataFrame, data_dir: Path) -> pd.DataFrame:
    """Drop records outside Massachusetts using a spatial polygon check."""
    import geopandas as gpd

    ma = load_massachusetts_boundary(data_dir)
    points = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    )

    filtered = gpd.sjoin(points, ma[["geometry"]], predicate="within", how="inner")
    filtered = filtered.drop(columns=["geometry", "index_right"], errors="ignore")
    return pd.DataFrame(filtered)



In [4]:

# -----------------------------------------------------------------------------
# 3. Main pipeline
# -----------------------------------------------------------------------------

def main() -> None:
    api_key = load_opencage_key()

    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")

    incidents = pd.read_csv(INPUT_CSV)
    if "Location" not in incidents.columns:
        raise ValueError("Input CSV must contain a 'Location' column.")

    raw_locations = (
        incidents["Location"]
        .dropna()
        .astype(str)
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    unique_locations = pd.DataFrame({"raw_address": raw_locations})
    unique_locations["cleaned_address"] = unique_locations["raw_address"].apply(clean_address)

    cache = load_cache(CACHE_CSV)
    cached_raw_addresses = set(cache["raw_address"].dropna().astype(str).str.strip())
    to_geocode = unique_locations[~unique_locations["raw_address"].isin(cached_raw_addresses)].copy()

    total_unique_locations = len(unique_locations)
    cache_hits = total_unique_locations - len(to_geocode)
    new_api_calls = 0

    print("Geocoding run started")
    print(f"Input rows: {len(incidents):,}")
    print(f"Total unique locations: {total_unique_locations:,}")
    print(f"Cache hits: {cache_hits:,}")
    print(f"New addresses to geocode: {len(to_geocode):,}")

    new_rows = []
    for i, row in enumerate(to_geocode.itertuples(index=False), start=1):
        try:
            result = geocode_with_opencage(row.cleaned_address, api_key)
            new_api_calls += 1
        except RuntimeError as e:
            if str(e) == "OPENCAGE_QUOTA_EXHAUSTED":
                if new_rows:
                    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
                    save_cache(cache, CACHE_CSV)
                print("\n" + "=" * 60)
                print("OPENCAGE DAILY REQUEST LIMIT REACHED")
                print(f"Processed this run: {i - 1:,}")
                print(f"Remaining uncached: {len(to_geocode) - (i - 1):,}")
                print(f"Progress saved to: {CACHE_CSV}")
                print("Rerun later and it will continue from the cache.")
                print("=" * 60)
                return

            raise

        new_rows.append(
            {
                "raw_address": row.raw_address,
                "cleaned_address": row.cleaned_address,
                "lat": result["lat"],
                "long": result["long"],
                "address_confidence": result["address_confidence"],
                "geocoded_at": datetime.now(timezone.utc).isoformat(),
            }
        )

        print(
            f"[{i:,}/{len(to_geocode):,}] {row.raw_address} -> "
            f"{result['lat']}, {result['long']} | confidence={result['address_confidence']}"
        )

        if i % BATCH_SAVE_EVERY == 0:
            cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
            save_cache(cache, CACHE_CSV)
            new_rows = []
            print(f"Cache checkpoint saved: {CACHE_CSV}")

        time.sleep(SLEEP_SECONDS)

    if new_rows:
        cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
        save_cache(cache, CACHE_CSV)

    cache = load_cache(CACHE_CSV)

    failed_geocodes = int(cache[cache["raw_address"].isin(unique_locations["raw_address"])] ["lat"].isna().sum())

    # Merge coordinates onto all incident rows using raw Location values.
    incidents["raw_address"] = incidents["Location"].astype(str).str.strip()
    merged = incidents.merge(cache, on="raw_address", how="left")

    merged = merged.rename(
        columns={
            "lat": "latitude",
            "long": "longitude",
            "address_confidence": "geocode_confidence",
        }
    )

    # Numeric conversion before filtering.
    merged["latitude"] = pd.to_numeric(merged["latitude"], errors="coerce")
    merged["longitude"] = pd.to_numeric(merged["longitude"], errors="coerce")
    merged["geocode_confidence"] = pd.to_numeric(merged["geocode_confidence"], errors="coerce")

    merged["coord_valid"] = (
        merged["latitude"].between(40, 48)
        & merged["longitude"].between(-74, -66)
    )

    rows_before_coord_filter = len(merged)
    coord_filtered = merged.dropna(subset=["latitude", "longitude"]).copy()
    coord_filtered = coord_filtered[coord_filtered["coord_valid"]].copy()
    rows_after_coord_filter = len(coord_filtered)

    # Spatial MA boundary filter.
    rows_before_ma_filter = len(coord_filtered)
    ma_filtered = filter_to_massachusetts(coord_filtered, DATA_DIR)
    rows_after_ma_filter = len(ma_filtered)

    rows_dropped_after_coordinate_filter = rows_before_coord_filter - rows_after_coord_filter
    rows_dropped_after_ma_filter = rows_before_ma_filter - rows_after_ma_filter

    # Keep final columns clean. raw_address/cleaned_address are useful audit columns;
    # remove them here if downstream code should only see original columns + lat/lon/confidence.
    final = ma_filtered.drop(columns=["lat", "long"], errors="ignore")
    final.to_csv(OUTPUT_CSV, index=False)

    print("\nRun summary")
    print("-----------")
    print(f"Total unique locations: {total_unique_locations:,}")
    print(f"Cache hits: {cache_hits:,}")
    print(f"New API calls: {new_api_calls:,}")
    print(f"Failed geocodes in current address universe: {failed_geocodes:,}")
    print(f"Rows dropped after coordinate filter: {rows_dropped_after_coordinate_filter:,}")
    print(f"Rows dropped after MA boundary filter: {rows_dropped_after_ma_filter:,}")
    print(f"Final rows saved: {len(final):,}")
    print(f"Output written to: {OUTPUT_CSV}")
    print(f"Cache written to: {CACHE_CSV}")



In [5]:
api_key = load_opencage_key()
print(f"Key length: {len(api_key)}")
print(api_key[:8] + "...")
import requests

key = load_opencage_key()

r = requests.get(
    "https://api.opencagedata.com/geocode/v1/json",
    params={
        "q": "Lawrence, MA",
        "key": key,
        "limit": 1,
        "no_annotations": 1,
    },
    timeout=30,
)

print("Status code:", r.status_code)
print(r.text[:1000])

Key length: 32
f8433c8c...
Status code: 402
{"documentation":"https://opencagedata.com/api","licenses":[{"name":"see attribution guide","url":"https://opencagedata.com/credits"}],"rate":{"limit":2500,"remaining":0,"reset":1782259200},"results":[],"status":{"become_a_customer":"https://opencagedata.com/pricing","code":402,"message":"quota exceeded"},"stay_informed":{"blog":"https://blog.opencagedata.com","mastodon":"https://en.osm.town/@opencage"},"thanks":"For using an OpenCage API","timestamp":{"created_http":"Tue, 23 Jun 2026 18:17:47 GMT","created_unix":1782238667},"total_results":0}


In [6]:

if __name__ == "__main__":
    main()


/var/folders/f6/0w5q0_md1413229qftcy9h840000gn/T/ipykernel_46435/3299209768.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  incidents = pd.read_csv(INPUT_CSV)


Geocoding run started
Input rows: 428,527
Total unique locations: 58,265
Cache hits: 25,171
New addresses to geocode: 33,094

OPENCAGE DAILY REQUEST LIMIT REACHED
Processed this run: 0
Remaining uncached: 33,094
Progress saved to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/geocode_cache.csv
Rerun later and it will continue from the cache.
